In [2]:
# Oxygen Results

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver

oxygen_driver = PySCFDriver(
    atom=[
        "O  0.000  0.000  0.000"
    ],
    unit=DistanceUnit.ANGSTROM,
    basis="sto3g",
    charge=0,
    spin=2
)

oxygen_problem = oxygen_driver.run()

print("Electrons:", oxygen_problem.num_particles)
print("Spatial orbitals:", oxygen_problem.num_spatial_orbitals)
print("Spin:", oxygen_problem.num_particles[0] - oxygen_problem.num_particles[1])
print("Nuclear repulsion:", oxygen_problem.nuclear_repulsion_energy)


Electrons: (5, 3)
Spatial orbitals: 5
Spin: 2
Nuclear repulsion: 0


In [3]:
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper

oxygen_transformer = ActiveSpaceTransformer(
    num_electrons=6,
    num_spatial_orbitals=4
)

oxygen_active_problem = oxygen_transformer.transform(
    oxygen_problem
)

oxygen_mapper = JordanWignerMapper()

oxygen_hamiltonian = oxygen_mapper.map(
    oxygen_active_problem.hamiltonian.second_q_op()
)

print("Active electrons:", oxygen_active_problem.num_particles)
print("Active orbitals:", oxygen_active_problem.num_spatial_orbitals)
print("Qubits:", oxygen_hamiltonian.num_qubits)

Active electrons: (4, 2)
Active orbitals: 4
Qubits: 8


In [4]:
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

# Create the oxygen Hartree-Fock state
oxygen_hf = HartreeFock(
    num_spatial_orbitals=oxygen_active_problem.num_spatial_orbitals,
    num_particles=oxygen_active_problem.num_particles,
    qubit_mapper=oxygen_mapper
)

# Create the UCCSD circuit
oxygen_ansatz = UCCSD(
    num_spatial_orbitals=oxygen_active_problem.num_spatial_orbitals,
    num_particles=oxygen_active_problem.num_particles,
    qubit_mapper=oxygen_mapper,
    initial_state=oxygen_hf
)

print("Qubits:", oxygen_ansatz.num_qubits)
print("Parameters:", oxygen_ansatz.num_parameters)
print("Circuit depth:", oxygen_ansatz.depth())

Qubits: 8
Parameters: 5
Circuit depth: 1


In [5]:
print("Number of excitations:", len(oxygen_ansatz.excitation_list))

print(
    "Decomposed circuit depth:",
    oxygen_ansatz.decompose(reps=5).depth()
)

print("Parameters:")
print(list(oxygen_ansatz.parameters))

Number of excitations: 5
Decomposed circuit depth: 161
Parameters:
[ParameterVectorElement(t[0]), ParameterVectorElement(t[1]), ParameterVectorElement(t[2]), ParameterVectorElement(t[3]), ParameterVectorElement(t[4])]


In [6]:
from qiskit_algorithms import NumPyMinimumEigensolver

oxygen_exact_solver = NumPyMinimumEigensolver()

oxygen_exact_result = oxygen_exact_solver.compute_minimum_eigenvalue(
    oxygen_hamiltonian
)

oxygen_classical_energy = float(
    oxygen_exact_result.eigenvalue.real
)

print(
    "Oxygen classical electronic energy:",
    oxygen_classical_energy,
    "Hartree"
)

print("\nHamiltonian constants:")
print(oxygen_active_problem.hamiltonian.constants)

Oxygen classical electronic energy: -15.354068156742722 Hartree

Hamiltonian constants:
{'nuclear_repulsion_energy': 0, 'ActiveSpaceTransformer': np.float64(-58.45008207651324)}


In [7]:
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

oxygen_exact_solver = NumPyMinimumEigensolver(
    filter_criterion=oxygen_active_problem.get_default_filter_criterion()
)

oxygen_ground_solver = GroundStateEigensolver(
    oxygen_mapper,
    oxygen_exact_solver
)

oxygen_exact_result = oxygen_ground_solver.solve(
    oxygen_active_problem
)

print("Verified oxygen total energy:")
print(oxygen_exact_result.total_energies)

print("Active electrons:", oxygen_active_problem.num_particles)
print("Required spin difference:", 2)

Verified oxygen total energy:
[-73.70926134]
Active electrons: (4, 2)
Required spin difference: 2


In [8]:
from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

ibm_token = getpass("Enter your IBM Quantum API key: ")

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=ibm_token
)

backend = service.backend("ibm_fez")

print("Connected to:", backend.name)


qiskit_runtime_service._resolve_cloud_instances:WARNING:2026-11-07 14:09:52,075: Default instance not set. Searching all available instances.


Connected to: ibm_fez


In [9]:
from qiskit.transpiler.preset_passmanagers import (
    generate_preset_pass_manager
)

oxygen_pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3
)

oxygen_isa_ansatz = oxygen_pm.run(oxygen_ansatz)

oxygen_isa_hamiltonian = oxygen_hamiltonian.apply_layout(
    oxygen_isa_ansatz.layout
)

print("Physical circuit width:", oxygen_isa_ansatz.num_qubits)
print("Circuit depth:", oxygen_isa_ansatz.depth())
print("Gate counts:", oxygen_isa_ansatz.count_ops())
print("Two-qubit gates:", oxygen_isa_ansatz.count_ops().get("cz", 0))
print("Parameters:", oxygen_isa_ansatz.num_parameters)

Physical circuit width: 156
Circuit depth: 255
Gate counts: OrderedDict({'rz': 128, 'sx': 116, 'cz': 72, 'x': 14})
Two-qubit gates: 72
Parameters: 5


In [11]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
import numpy as np

oxygen_estimator = Estimator(mode=backend)
oxygen_estimator.options.default_shots = 4096


oxygen_initial_parameters = np.zeros(
    oxygen_isa_ansatz.num_parameters
)

print("Oxygen parameters:", oxygen_initial_parameters)
print("Ready to submit to IBM hardware.")

Oxygen parameters: [0. 0. 0. 0. 0.]
Ready to submit to IBM hardware.


In [12]:
print("Submitting oxygen calculation...")

oxygen_job = oxygen_estimator.run([
    (
        oxygen_isa_ansatz,
        oxygen_isa_hamiltonian,
        oxygen_initial_parameters
    )
])

print("Job submitted!")
print("Job ID:", oxygen_job.job_id())

Submitting oxygen calculation...
Job submitted!
Job ID: dbnj2kc4148s73914frg


In [19]:
print(oxygen_job.status())

QUEUED


In [15]:
oxygen_result = oxygen_job.result()

oxygen_hardware_electronic = float(
    oxygen_result[0].data.evs
)

print("Oxygen electronic energy:", oxygen_hardware_electronic, "Hartree")

KeyboardInterrupt: 

In [16]:
oxygen_energy_offset = sum(
    float(value)
    for value in oxygen_active_problem.hamiltonian.constants.values()
)

oxygen_hardware_total = (
    oxygen_hardware_electronic + oxygen_energy_offset
)

oxygen_classical_total = float(
    oxygen_exact_result.total_energies[0]
)

print("IBM oxygen total energy:", oxygen_hardware_total, "Hartree")
print("Classical oxygen total energy:", oxygen_classical_total, "Hartree")
print(
    "Difference:",
    abs(oxygen_hardware_total - oxygen_classical_total),
    "Hartree"
)

NameError: name 'oxygen_hardware_electronic' is not defined